# Building a stock-research harness

**Lecture 10 · Special Topics · November 5, 2026**

In Lectures 2 and 4, we turned holdings and prices into stock reports. We knew which calculations to run and which inputs they needed. This notebook adds a model that can choose what to investigate, call Python functions, inspect the results, and write an interpretation.

We will build the surrounding application as well. It needs to keep useful evidence, distinguish an open question from a fact, resume work after an interruption, and decide when a result is ready for the report. We will use Pydantic AI Harness and the Python package in `research_harness/` to make those decisions concrete.

The notebook follows the content of the slide deck with fuller explanations and executable experiments. Read each prediction question before running the next cell. The presenter can follow the live route in [presenter_notes.md](presenter_notes.md). The [Python scripts](readme.md#code-map) remain available for running the application outside Jupyter.

| Part | Sections | What we investigate |
|---|---|---|
| Agent and tools | 1–6 | Model decisions, Python execution, contracts, validation, effects |
| Memory and context | 7–12 | Write policies, dates, retrieval, context limits, concurrent edits |
| Longer projects | 13–15 | Checkpoints, uncertain writes, budgets, plans |
| Research output | 16–18 | Report acceptance, evaluation, another research module |

The core route takes three hours, including two ten-minute breaks. The final extensions are available for independent practice.

## 0. Environment and working directory

From the Lecture 10 folder, run these commands in a terminal:

```sh
uv sync --locked --group notebook
uv run --group notebook jupyter lab
```

Open this notebook and use the Python kernel from Lecture 10's `.venv`. In VS Code, choose that environment in the kernel selector. The first cell checks the versions and locates the lecture folder whether the kernel starts here or at the repository root.

The examples use the included price and source files. Their construction is described in the [data guide](data/README.md). The default route uses `FunctionModel` to make the model's choices predictable while the actual Pydantic agent executes its tools. At the end, a separate opt-in cell can run the same application with a configured provider.

Each full execution creates a new directory under `runs/`. Individual experiments also choose fresh names, so rerunning them does not overwrite an earlier project. The notebook uses top-level `await` for agent and memory operations, which [IPython supports](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html).

In [1]:
# S0 setup
import asyncio
import contextlib
from dataclasses import dataclass
import hashlib
import io
import json
import os
from pathlib import Path
import sqlite3
import subprocess
import sys
from typing import Annotated
from uuid import uuid4
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.messages import ModelResponse, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import FunctionModel
from pydantic_ai.usage import UsageLimits

candidates = [Path.cwd(), Path.cwd() / "Lectures" / "Lecture 10"]
LECTURE_DIR = next((p.resolve() for p in candidates if (p / "research_harness").is_dir()), None)
if LECTURE_DIR is None:
    raise FileNotFoundError("Start the kernel in Lecture 10 or the repository root.")
sys.path.insert(0, str(LECTURE_DIR)) if str(LECTURE_DIR) not in sys.path else None
os.environ["PYDANTIC_AI_NO_BANNER"] = "1"
pd.set_option("display.max_colwidth", 100)

RUN_ROOT = LECTURE_DIR / "runs" / f"notebook-{uuid4().hex[:8]}"
RUN_ROOT.mkdir(parents=True)

def fresh_path(label):
    return RUN_ROOT / f"{label}-{uuid4().hex[:8]}"

def run_script(script, *arguments, expected=(0,), timeout=180):
    """Use this kernel's Python for process-boundary demonstrations."""
    completed = subprocess.run(
        [sys.executable, str(LECTURE_DIR / script), *map(str, arguments)],
        cwd=LECTURE_DIR, capture_output=True, text=True, timeout=timeout,
    )
    if completed.stdout:
        print(completed.stdout.replace(str(LECTURE_DIR), "Lecture 10"))
    if completed.returncode not in expected:
        raise RuntimeError(completed.stderr or completed.stdout)
    return completed

required = {"pydantic-ai-harness": "0.65.0", "pydantic-ai-slim": "2.65.0"}
installed = {name: version(name) for name in required}
assert installed == required, "Select Lecture 10's environment and run uv sync --locked --group notebook."
display(pd.Series(installed, name="Installed version").to_frame())
print("Project outputs:", RUN_ROOT.relative_to(LECTURE_DIR))

,Installed version
pydantic-ai-harness,0.65.0
pydantic-ai-slim,2.65.0


Project outputs: runs/notebook-6ddeb90e


### A calculation we can predict

Prices move from 100 to 110 to 99. The daily returns are 10% and −10%, but the whole-period return is −1%. The second percentage applies to a different starting value. We will keep calculations like this in Python and let the model choose when it needs them.

**Before running:** what is the maximum drawdown, and why is it different from the total return?

In [2]:
# S1 return arithmetic
prices = pd.Series([100.0, 110.0, 99.0], name="price")
arithmetic = pd.DataFrame({
    "price": prices,
    "simple_return": prices.pct_change(fill_method=None),
    "log_return": np.log(prices / prices.shift(1)),
    "drawdown": prices / prices.cummax() - 1,
})
display(arithmetic)
print(f"Total return: {prices.iloc[-1] / prices.iloc[0] - 1:.1%}")
print(f"Maximum drawdown: {arithmetic.drawdown.min():.1%}")
assert np.isclose(prices.iloc[-1] / prices.iloc[0] - 1, -.01)

,price,simple_return,log_return,drawdown
0,100.0,NaN,NaN,0.0
1,110.0,0.1,0.095310,0.0
2,99.0,-0.1,-0.105361,-0.1


Total return: -1.0%
Maximum drawdown: -10.0%


In [3]:
# S2 portfolio evidence
from research_harness.data import risk_metrics, portfolio_exposure

price_frame = pd.read_csv(LECTURE_DIR / "data/prices.csv", parse_dates=["date"])
metrics = {ticker: risk_metrics(group.sort_values("date"))
           for ticker, group in price_frame.groupby("ticker")}
exposure = portfolio_exposure(LECTURE_DIR / "data/holdings.csv", metrics, cash=5000)
display(pd.DataFrame(exposure["positions"]))
print(f"Cash weight: {exposure['cash_weight']:.2%}")
assert np.isclose(sum(p["weight"] for p in exposure["positions"]) + exposure["cash_weight"], 1)

,ticker,shares,cost,weighted_entry_price,raw_close,market_value,weight,unrealized_pnl
0,AAPL,60,5700,95.000000,101.399142,6083.94852,0.311582,383.94852
1,JPM,60,3300,55.000000,51.391579,3083.49474,0.157917,-216.50526
2,MSFT,60,4570,76.166667,89.309461,5358.56766,0.274432,788.56766


Cash weight: 25.61%


The stock weights and cash weight sum to one. Dropping cash would change every equity weight. Later, dropping a failed holding would cause the same kind of denominator error. These are application rules we can check directly.

## 1. What happens inside an agent loop?

A model request contains the context available at that moment: instructions, conversation messages, tool definitions, and any retrieved material we have included. The response may contain an answer, a tool call, or several tool calls. A tool call is a request to execute a function with particular arguments.

For example, the model might produce this request:

```json
{
  "tool_name": "search_sources",
  "args": {"query": "debt refinancing", "limit": 2},
  "tool_call_id": "call_17"
}
```

The application validates the arguments, runs the function, and sends the result back with the same call ID. That association matters when a response contains several calls. The next model request can use the returned evidence to choose another action or produce a final answer.

| Stage | What happens | A failure we need to handle |
|---|---|---|
| Construct context | Select instructions, history, schemas, and evidence | Relevant information never reaches the model |
| Request a response | Call the model provider | Timeout or rate limit |
| Validate a call | Check the function name and argument schema | Missing field or invalid range |
| Execute Python | Check scope, access data, calculate or write | Missing data or a partially completed effect |
| Return an observation | Serialize a result with its call ID | Output is too large or loses its units |
| Continue or finish | Make another request or validate the final output | Repeated work or an unsupported conclusion |

We can describe the loop as a sequence of contexts, actions, and observations. At step `t`, the model sees `context_t` and proposes `action_t`. The runtime executes the action and obtains `observation_t`. Our context policy decides which parts enter `context_(t+1)`.

This last step is easy to overlook. Saving a file does not make its contents available to every future model request. Something must retrieve the file and include its contents. Persistent memory changes future inputs to the model; it does not update the model's trained parameters.

The first agent example below shows the smallest version of this loop. Its `FunctionModel` produces predictable responses, but Pydantic AI still performs the tool dispatch and output validation. This lets us study the runtime without variability in the model's decisions. Later, `--engine model` uses a provider-backed model with the same application code. The [Pydantic AI tool documentation](https://pydantic.dev/docs/ai/tools/) describes how Python functions become tool definitions.

### A tool call we can inspect

We will supply predictable model responses for this first run. The first response requests `total_return`; the second uses the returned observation to construct a `ReturnSummary`. The runtime still generates the tool schema, validates the call, executes the function, and validates the final output.

**Before running:** which line calculates the return? Which line proposes the tool call?

In [4]:
# T1 first agent
class ReturnSummary(BaseModel):
    total_return: float
    explanation: str


def classroom_model(messages, info):
    observations = [p for m in messages for p in m.parts if isinstance(p, ToolReturnPart)]
    if not observations:
        return ModelResponse(parts=[ToolCallPart(
            "total_return", {"first": 100, "last": 99}, tool_call_id="return-1"
        )])
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, {
        "total_return": observations[-1].content,
        "explanation": "The final price is 1% below the starting price.",
    })])


first_agent = Agent(FunctionModel(classroom_model), output_type=ReturnSummary)


@first_agent.tool_plain
def total_return(first: float, last: float) -> float:
    """Calculate the whole-period return from positive endpoint prices."""
    if first <= 0 or last <= 0:
        raise ModelRetry("Use positive endpoint prices.")
    return last / first - 1

first_result = await first_agent.run("Prices went from 100 to 110 to 99. What was the total return?")
display(first_result.output.model_dump())

{'total_return': -0.010000000000000009,
 'explanation': 'The final price is 1% below the starting price.'}

In [5]:
# T2 message trace
def message_trace(result):
    rows = []
    for message in result.all_messages():
        for part in message.parts:
            if isinstance(part, (ToolCallPart, ToolReturnPart)):
                rows.append({
                    "kind": type(part).__name__,
                    "call_id": part.tool_call_id,
                    "tool": part.tool_name,
                    "content": str(part.args if isinstance(part, ToolCallPart) else part.content),
                })
    return pd.DataFrame(rows)

display(message_trace(first_result))
print("Model requests:", first_result.usage.requests)
print("Successful tool calls:", first_result.usage.tool_calls)

,kind,call_id,tool,content
0,ToolCallPart,return-1,total_return,"{'first': 100, 'last': 99}"
1,ToolReturnPart,return-1,total_return,-0.010000000000000009
2,ToolCallPart,pyd_ai_a3e102d70cd64480b90097a02090e40f,final_result,"{'total_return': -0.010000000000000009, 'explanation': 'The final price is 1% below the starting..."
3,ToolReturnPart,pyd_ai_a3e102d70cd64480b90097a02090e40f,final_result,Final result processed.


Model requests: 2
Successful tool calls: 1


Follow `return-1` from the call to the observation. A call ID associates a result with the request that produced it. The structured final answer also uses an output-tool representation in this run, but the successful function-tool count measures the executed research function. We will separate these counters more carefully in the validation experiment.

## 2. Where the harness ends and our application begins

Pydantic AI runs the model/tool loop. Its harness capabilities provide planning, working memory, conversation persistence, and context management. Our application supplies the financial calculations, source corpus, task queue, acceptance checks, and report.

In `research_harness/agent.py`, we compose the agent with a list of capabilities. Here is a shortened version of that configuration:

```python
capabilities = [
    Planning(store=plan_store, tools=["write_plan"]),
    Memory(memory_store, namespace=ticker, max_tokens=1200),
    SlidingWindowCompaction(max_tokens=24000, keep_tokens=12000),
    StepPersistence(store=step_store, run_id=attempt_id),
]
```

Each capability has a particular job. A saved plan helps the model organize an investigation. Working memory retains useful notes. A checkpoint helps the runtime restore a conversation. None of these, by itself, establishes that a ticker has a valid research result.

The supervisor owns that decision. It assigns a ticker, reserves an attempt, starts a worker, and checks for an accepted note. The model may mark every item in its plan complete, but the task remains incomplete if its final citations fail validation. Keeping those responsibilities separate gives us a clear answer to “who decides that the work is done?” See the [harness overview](https://pydantic.dev/docs/ai/harness/) for the capability interfaces.

## 3. Designing a tool contract

A useful tool interface expresses an action the model can choose while leaving application constraints in Python. Our search tool accepts a query and a small result limit:

```python
@agent.tool
def search_sources(
    ctx: RunContext[ResearchDeps],
    query: Annotated[str, Field(min_length=1, max_length=200)],
    limit: Annotated[int, Field(ge=1, le=5)] = 3,
) -> dict:
    d = ctx.deps
    return dispatch(d.run, d.ticker, d.attempt, "search_sources",
                    {"query": query, "limit": limit})
```

The model controls the search terms. Python supplies the project, assigned ticker, and attempt through `ResearchDeps`. The project's configuration supplies the research date. We do not need the model to repeat these constraints correctly on every call. `RunContext` carries application dependencies and does not appear as a model-supplied parameter in the tool schema. See [dependencies](https://pydantic.dev/docs/ai/dependencies/).

Before implementing a tool, write its contract in ordinary language:

| Decision | Example for `search_sources` |
|---|---|
| Purpose | Find excerpts relevant to a particular research question |
| Model arguments | Query of 1–200 characters, result limit of 1–5 |
| Application scope | Current project, assigned ticker, fixed publication cutoff |
| Preconditions | Input hashes still match the saved project |
| Result | Whole evidence records, citation IDs, matched and omitted counts |
| Side effect | Save returned evidence and a retrieval event |
| Retry behavior | Identical evidence reuses its content-derived ID |
| Completion meaning | A search result supplies evidence but does not complete research |

Tool granularity is a design choice. A general `run_sql(sql)` tool gives the model substantial flexibility, but requires database permissions, query limits, and a careful definition of allowed data. A separate tool for every arithmetic operation makes the model spend requests orchestrating calculations we already know how to perform. Here, `analyze_prices` performs a coherent calculation and returns its conventions and coverage. The model chooses when to request that analysis and how to interpret it.

Descriptions matter as well as types. “Search documents” gives little guidance. “Search eligible excerpts for the assigned ticker using lexical overlap” tells the model both the scope and the search behavior. Describe empty results, truncation, and units explicitly. Those details affect what the model should do next.

### Application scope in a runnable tool

Here the model supplies a query. The dependency object supplies the ticker, publication cutoff, and corpus. This smaller search agent lets us inspect that boundary without creating a full project yet. Its retrieved records are ordinary source records; the full project's dispatcher adds persistent evidence IDs and usage receipts.

In [6]:
# T3 dependency scope
from research_harness.retrieval import rank_sources, pack_evidence, terms
from research_harness.memory import evidence_record

corpus = json.loads((LECTURE_DIR / "data/retrieval_lab_sources.json").read_text())

@dataclass
class SourceScope:
    ticker: str
    as_of: str
    records: list[dict]

class SearchSummary(BaseModel):
    record_ids: list[str]

search_schemas = []

def search_model(messages, info):
    search_schemas[:] = [info.function_tools[0].parameters_json_schema]
    observed = [p for m in messages for p in m.parts if isinstance(p, ToolReturnPart)]
    if not observed:
        return ModelResponse(parts=[ToolCallPart("find_sources", {"query": "debt refinancing"})])
    ids = [record["record_id"] for record in observed[-1].content]
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, {"record_ids": ids})])

search_agent = Agent(FunctionModel(search_model), deps_type=SourceScope, output_type=SearchSummary)

@search_agent.tool
def find_sources(ctx: RunContext[SourceScope], query: Annotated[str, Field(min_length=1)]) -> list[dict]:
    """Find excerpts by lexical overlap within the assigned ticker and cutoff."""
    scope = ctx.deps
    _, ranked = rank_sources(scope.records, ticker=scope.ticker, as_of=scope.as_of, query=query)
    return ranked[:2]

search_result = await search_agent.run(
    "Find the debt discussion.", deps=SourceScope("AAPL", "2024-04-30", corpus)
)
display(search_result.output.model_dump())
display(search_schemas[0])
assert set(search_schemas[0]["properties"]) == {"query"}

{'record_ids': ['aapl-debt']}

{'additionalProperties': False,
 'properties': {'query': {'minLength': 1, 'type': 'string'}},
 'required': ['query'],
 'type': 'object'}

The schema exposes `query`; it does not expose `ctx`, `ticker`, or `as_of`. Change the dependency's ticker and rerun to see that the scope comes from Python. A provider-backed model may choose different query terms, but it receives the same interface.

## 4. Validation happens at several boundaries

A type annotation handles only part of a tool's contract. We need to distinguish four checks:

1. **Schema validation:** Are the required arguments present? Do their types and ranges satisfy the schema?
2. **Semantic validation:** Is this request meaningful and permitted for this project? Is there enough data for the requested window?
3. **Result validation:** Are the values finite? Are the dates, units, and identifiers intact?
4. **Completion validation:** Does the final research note cite the required evidence for the assigned ticker?

Pydantic can coerce some inputs to the requested type unless we choose strict validation. A value reaching a Python function has passed the configured schema, but we still need the semantic checks inside the function. A schema is also not an authorization system. The application must enforce data scope independently of what the prompt asks the model to do.

The validation experiment below traces the same mechanism as `05_tool_call_lifecycle.py`. The model requests the same calculation three times:

| Requested window | Schema | Function entered? | Result |
|---|---|---|---|
| 1 observation | Fails the minimum of 2 | No | Validation feedback |
| 12 observations | Passes | Yes | `ModelRetry`: only 3 observations exist |
| 3 observations | Passes | Yes | Total return of −1% |

The trace shows the original call IDs beside the responses. The calculation runs once, although the Python tool function is entered twice. The model makes four requests: three tool requests followed by a structured answer.

There is a useful accounting detail here. In the pinned SDK, `RunUsage.tool_calls` counts **successful** tool calls. This experiment therefore records four requests and one successful tool call. A tool-call limit alone would not bound repeated invalid requests. We also set a request limit, and our application ledger reserves dispatched research actions before executing them.

Use `ModelRetry` when the model can repair its request or output from the feedback. An unsupported window or unknown citation is a good example. A database outage usually calls for a runtime retry or a failed task, since changing the wording of the query will not restore the database. Returning an empty list for both “no matches” and “service unavailable” would hide a distinction the research process needs.

In [7]:
# T4 validation setup
class Calculation(BaseModel):
    total_return: float
    observations: int


def build_validation_experiment():
    entered, calculated, schemas = [], [], []
    available_prices = [100.0, 110.0, 99.0]

    def scripted_requests(messages, info):
        schemas[:] = [info.function_tools[0].parameters_json_schema]
        observations = [p for m in messages for p in m.parts if isinstance(p, ToolReturnPart)]
        requests = [(1, "schema-error"), (12, "coverage-error"), (3, "valid-request")]
        if len(observations) < len(requests):
            window, call_id = requests[len(observations)]
            return ModelResponse(parts=[ToolCallPart(
                "calculate_return", {"window": window}, tool_call_id=call_id
            )])
        return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, observations[-1].content)])

    agent = Agent(FunctionModel(scripted_requests), output_type=Calculation, retries=3)

    @agent.tool_plain
    def calculate_return(window: Annotated[int, Field(ge=2, le=252)]) -> dict:
        """Calculate return over the final window price observations."""
        entered.append(window)
        if window > len(available_prices):
            raise ModelRetry(f"Only {len(available_prices)} prices exist. Use a window from 2 to 3.")
        selected = available_prices[-window:]
        value = {"total_return": selected[-1] / selected[0] - 1, "observations": len(selected)}
        calculated.append(value)
        return value

    return agent, entered, calculated, schemas

**Pause before T5:** predict the number of model requests, function entries, and completed calculations. The requested windows are 1, 12, and 3, followed by the final answer. Read the placement of `entered.append` relative to the semantic check.

In [8]:
# T5 validation trace
validation_agent, entered, calculated, schemas = build_validation_experiment()
validation_result = await validation_agent.run(
    "Calculate the return over the available history.",
    usage_limits=UsageLimits(request_limit=5, tool_calls_limit=3),
)
display(message_trace(validation_result))
display(pd.Series({
    "model_requests": validation_result.usage.requests,
    "function_entries": len(entered),
    "completed_calculations": len(calculated),
    "successful_tool_calls": validation_result.usage.tool_calls,
}).to_frame("count"))
display(schemas[0]["properties"]["window"])
assert entered == [12, 3] and len(calculated) == 1
assert validation_result.usage.requests == 4 and validation_result.usage.tool_calls == 1

,kind,call_id,tool,content
0,ToolCallPart,schema-error,calculate_return,{'window': 1}
1,ToolReturnPart,schema-error,calculate_return,"[{'type': 'greater_than_equal', 'loc': ['window'], 'msg': 'Input should be greater than or equal..."
2,ToolCallPart,coverage-error,calculate_return,{'window': 12}
3,ToolReturnPart,coverage-error,calculate_return,Only 3 prices exist. Use a window from 2 to 3.
4,ToolCallPart,valid-request,calculate_return,{'window': 3}
5,ToolReturnPart,valid-request,calculate_return,"{'total_return': -0.010000000000000009, 'observations': 3}"
6,ToolCallPart,pyd_ai_08f2cf09732d42c2bf2a9e7748c0a794,final_result,"{'total_return': -0.010000000000000009, 'observations': 3}"
7,ToolReturnPart,pyd_ai_08f2cf09732d42c2bf2a9e7748c0a794,final_result,Final result processed.


,count
model_requests,4
function_entries,2
completed_calculations,1
successful_tool_calls,1


{'maximum': 252, 'minimum': 2, 'type': 'integer'}

## 5. Designing the result the model will read

Consider a tool returning `0.3029`. We cannot tell whether this is a return, a volatility, or a portfolio weight. Even “volatility: 0.3029” leaves open the frequency, price adjustment, date window, and estimator.

Our risk evidence includes the assigned ticker, requested and observed dates, row count, and calculation conventions. Annualized volatility uses the sample standard deviation of daily log returns multiplied by the square root of 252. Total return uses adjusted endpoint prices. Portfolio valuation uses raw close because it values the shares we actually hold. These choices reproduce the analysis from Lectures 2 and 4.

Source excerpts need comparable care. If a filing table says “USD millions,” that header belongs with the extracted values. A chunk containing “Debt: 100” without the header can create an error of a millionfold. A useful excerpt carries the company, reporting period, publication date, document locator, and enough surrounding text to interpret the number.

Large results create another problem. Sending every document on every call increases latency and consumes context. Cutting a serialized JSON string halfway through a record can remove a citation ID, unit, or closing delimiter. Our `pack_evidence` function includes whole records within a character budget and returns `matched`, `omitted`, and `limit` alongside the records. If a record does not fit, it remains in the corpus and contributes to the omitted count.

The caller can then narrow the query or use `read_source`. An empty search response means that this retrieval policy found no eligible matches. It does not establish that the company has no debt or that a fact does not exist elsewhere.

In [9]:
# T6 result interpretation
aapl_metrics = metrics["AAPL"]
display(pd.Series(aapl_metrics, name="Value").to_frame())
print("Ambiguous value alone:", round(aapl_metrics["annualized_log_volatility"], 4))
print(f"Annualized log-return volatility: {aapl_metrics['annualized_log_volatility']:.2%}")

,Value
first_date,2024-01-02
last_date,2024-04-30
observations,86
total_return,0.013991
annualized_log_volatility,0.302859
mean_rolling_20_log_volatility,0.297494
max_drawdown,-0.115432
approximate_sharpe_rf_2pct,0.069976
last_raw_close,101.399142
conventions,Daily log returns; sample std; 252 sessions; 2% annual risk-free approximation


Ambiguous value alone: 0.3029
Annualized log-return volatility: 30.29%


## 6. Side effects, retries, and parallel calls

Reading a local immutable file is straightforward to retry. Appending a note, submitting a job, or updating an external service introduces a commit boundary. If a tool times out, the caller may not know whether the effect happened.

The write experiment later in this notebook reproduces `07_retries_and_effects.py` one transaction at a time. Its first writer inserts a note, commits the transaction, and then raises a timeout to represent a lost acknowledgement. Retrying creates a second note. The failure occurred after the effect, so retrying the function repeated the effect.

The second writer uses a stable operation key, such as `project-17:AAPL:debt-question:v1`. It saves the note and a receipt for that key in the same transaction. When the caller retries with the same key and arguments, the function returns the existing receipt. Reusing the key with different arguments raises an error.

This is **idempotency**: repeating a particular operation has the same intended effect as executing it once. A model-generated tool-call ID is useful for connecting a request and response, but a new attempt can generate a new call ID. The operation key should identify the business action we want to deduplicate across those attempts.

The transaction in this example covers both the effect and its receipt because both live in one database. A local receipt cannot make a remote API call atomic with our local commit. For a remote write, use the service's idempotency support when available, or reconcile its state before repeating an uncertain operation. A durable workflow engine still needs a policy for this boundary.

Our application uses several different write behaviors. Evidence uses a canonical content hash and `INSERT OR IGNORE`, so an identical observation reuses an ID. An accepted note replaces the prior accepted note for that ticker. A working-memory edit can conflict with another edit, and blindly appending the same sentence twice would still create duplication. Each write deserves its own retry policy.

Parallel tool execution also changes the design. Independent reads can often overlap. A ratio calculation must wait for its inputs, and concurrent writes to the same note can overwrite one another. `async` permits overlapping work but does not make shared state safe. The course supervisor runs ticker tasks serially and holds a project lock. If we later parallelize it, we must revisit task claiming, provider rate limits, and shared report generation.

A cache solves a different problem: avoiding repeated work when inputs are unchanged. A risk-calculation cache key should include ticker, window, price-adjustment convention, source hash, and calculation version. A key containing only the ticker can return last month's result for this month's question. A tool-call ID, an idempotency key, and a cache key each identify something different.

### A reproducible cache identity

A persistent cache needs deterministic serialization. Python's built-in `hash()` is not a persistence format across interpreter sessions. Here we use canonical JSON and SHA-256. We include the calculation version because a change from log returns to simple returns changes the meaning of the result even if the input file stays the same.

In [10]:
# T7 cache identity
from research_harness.data import fingerprint

def cache_key(inputs):
    encoded = json.dumps(inputs, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()

cache_inputs = {
    "ticker": "AAPL", "start": "2024-01-02", "as_of": "2024-04-30",
    "adjustment": "adjusted_close", "source_sha256": fingerprint(LECTURE_DIR / "data/prices.csv"),
    "calculation_version": "log-volatility-v1",
}
original_key = cache_key(cache_inputs)
changed_key = cache_key({**cache_inputs, "start": "2024-02-01"})
display(pd.DataFrame([
    {"case": "original", "key": original_key},
    {"case": "same fields in another order", "key": cache_key(dict(reversed(list(cache_inputs.items()))))},
    {"case": "different start", "key": changed_key},
]))
assert original_key != changed_key

,case,key
0,original,2497a0bdb8711e27c7d5f351725d09248e02e0e98c675683be94668179815689
1,same fields in another order,2497a0bdb8711e27c7d5f351725d09248e02e0e98c675683be94668179815689
2,different start,96df5663b3528a9ac1a101a65cf845b426961fc6160deba6fb7d3ff0e79d99cc


### Break: 10 minutes

When we return, keep this question in mind: what must survive if we close the kernel right now? A saved answer, the reason for it, and the application's progress record serve different purposes.

## 7. What do we mean by memory?

“Add memory” is too broad to implement. Start by asking what the application must retain and how it will use it.

| Kind of memory | Example | How we use it |
|---|---|---|
| Current context | Recent tool results and instructions | Supply the next model request |
| Working memory | “Check whether cash covers near-term maturities” | Resume an investigation without rereading everything |
| Evidence | A dated source excerpt or exact risk calculation | Support claims and reproduce calculations |
| Plan | “Read debt schedule, compare cash, write note” | Organize the model's next steps |
| Workflow state | AAPL running, attempt 2, JPM pending | Decide what Python should execute next |
| Checkpoint | Settled conversation messages | Restore an interrupted agent run |
| Procedures | Tool definitions and research instructions | Define how research should proceed |

These are logical responsibilities. They do not require seven databases. Our application puts evidence, accepted notes, task status, and events in `memory.sqlite`. Pydantic AI Harness stores plans in `plans.sqlite`, working notes in `notebook.sqlite`, and conversation checkpoints in `steps.sqlite`.

M1 and M2 below create a project with one completed ticker and inspect its stores. Find an exact metric, an open research question, a plan item, and an accepted task. Ask which of those would be sufficient to reconstruct a report. A sentence in working memory saying “AAPL is done” cannot replace the task's acceptance checks.

A memory system needs a full lifecycle: decide what to write, attach metadata, store it, retrieve it, include it in context, and eventually update or retire it. Storage is only one step. A perfectly durable record that we never retrieve has no influence on the model's next answer.

### A project with one accepted ticker

We will create a project, run one task, and inspect the state that Python saved. The remaining tickers stay pending. The CLI returns 2 for an incomplete report, which is the expected result here.

These cells execute the application in a separate process using the notebook kernel's interpreter. That makes the saved state visible across process boundaries and keeps the application's scripts usable outside Jupyter.

In [11]:
# M1 one accepted ticker
project = fresh_path("portfolio")
run_script("research.py", "init", project)
run_script("research.py", "run", project, "--max-tasks", "1", expected=(2,))

from research_harness.memory import connect, evidence

def task_table(path):
    with contextlib.closing(connect(path)) as db:
        return pd.read_sql_query("SELECT ticker,status,attempts,error FROM tasks ORDER BY ticker", db)

display(task_table(project))
assert task_table(project).set_index("ticker").loc["AAPL", "attempts"] == 1

Created Lecture 10/runs/notebook-6ddeb90e/portfolio-eeebdf7a. Run: python research.py run Lecture 10/runs/notebook-6ddeb90e/portfolio-eeebdf7a



AAPL: attempt 1
INCOMPLETE: Lecture 10/runs/notebook-6ddeb90e/portfolio-eeebdf7a/report.html



,ticker,status,attempts,error
0,AAPL,complete,1,None
1,JPM,pending,0,None
2,MSFT,pending,0,None


In [12]:
# M2 inspect the stores
from pydantic_ai_harness.memory import SqliteMemoryStore, MemoryConflictError
from pydantic_ai_harness.planning import SqlitePlanStore
from pydantic_ai_harness.step_persistence import SqliteStepStore

with contextlib.closing(connect(project)) as db:
    aapl_evidence = evidence(db, "AAPL")
display(pd.DataFrame(aapl_evidence)[["id", "kind", "ticker"]])

plan = await SqlitePlanStore(str(project / "plans.sqlite"), session="AAPL").get_items()
display(pd.DataFrame([{"step": item.content, "status": item.status} for item in plan]))
working_store = SqliteMemoryStore(database=project / "notebook.sqlite")
working_note = await working_store.read("AAPL/main/MEMORY.md", max_chars=8000)
print("Working memory:", working_note.content)
step_store = SqliteStepStore(database=project / "steps.sqlite")
runs = await step_store.list_runs(conversation_id="AAPL")
snapshot = await step_store.latest_snapshot(run_id=runs[0].run_id)
print("Checkpoint:", runs[0].run_id, snapshot.state, "messages:", len(snapshot.messages))

,id,kind,ticker
0,94fd689e4b31e5167fb5,metrics,AAPL
1,e8bcca78a5f069b81059,source,AAPL


,step,status
0,Collect price and source evidence,TaskStatus.completed
1,Write findings and a follow-up question,TaskStatus.completed


Working memory: Next for AAPL: compare the latest eligible filing's business risks with the price history.

Checkpoint: AAPL-attempt-1 complete messages: 15


## 8. What should enter memory?

We should distinguish observations, sourced claims, and hypotheses. “The calculation returned 30.29% annualized volatility over this window” is an observation. “Management expects a stronger second half” is a claim attributable to a source. “Refinancing may become more expensive” is a hypothesis to investigate. Saving all three as unqualified facts encourages later requests to treat speculation as established evidence.

A richer evidence record might contain:

```python
{
    "record_id": "...",
    "project_id": "...",
    "entity": "AAPL",
    "kind": "reported_debt",
    "value": 100,
    "unit": "USD million",
    "period_end": "2023-12-31",
    "available_at": "2024-02-02",
    "ingested_at": "2024-02-03",
    "source_hash": "...",
    "locator": "document URL and page",
    "status": "reported",
    "supersedes": None,
}
```

This is a design example for a financial-statement extension. The current evidence schema stores `kind`, ticker, a content-derived ID, and a payload containing the relevant metadata. Our projects freeze their inputs and code, which keeps the first implementation manageable.

The dates answer different questions. `period_end` tells us when the measured economic period ended. `available_at` tells us when an investor could have known the information. `ingested_at` tells us when our system acquired it. A December 31 balance-sheet value published February 2 cannot enter a January research project just because the economic period ended in December.

Updates need a policy too. Suppose a February document reports debt of 100, and a May correction reports 120. An April investigation should retain the original observation. A June investigation should see the correction and its relationship to the original. Globally marking the old record “superseded” and filtering it out of every query would erase the historical view. We need to apply version and availability rules relative to the question's date.

Expiration depends on the information. A current quote may become stale quickly. A historical filing excerpt remains useful when its publication date and version are intact. A derived ratio becomes invalid when an input or calculation changes, even if only a minute has passed. A single time-to-live for all memory would miss these differences.

### A correction without erasing the earlier view

The two records below describe the same measure and reporting period. The second explicitly corrects the first. For this small example, we select the latest eligible version. A larger statement model would group by entity, measure, unit, and period before applying the version relationship.

**Before running:** should the April query return 100 or 120? What changes in June?

In [13]:
# M3 dated corrections
debt_versions = pd.DataFrame([
    {"id": "debt-v1", "entity": "AAPL", "value": 100, "unit": "USD million",
     "period_end": "2023-12-31", "available_at": "2024-02-02", "ingested_at": "2024-02-03",
     "source": "classroom/original", "supersedes": None},
    {"id": "debt-v2", "entity": "AAPL", "value": 120, "unit": "USD million",
     "period_end": "2023-12-31", "available_at": "2024-05-10", "ingested_at": "2024-05-11",
     "source": "classroom/correction", "supersedes": "debt-v1"},
])

def debt_known_on(cutoff):
    eligible = debt_versions.loc[debt_versions.available_at <= cutoff]
    if eligible.empty:
        return None
    return eligible.sort_values("available_at").iloc[-1].to_dict()

display(debt_versions)
display(pd.DataFrame([{"question_date": d, **debt_known_on(d)} for d in ["2024-04-30", "2024-06-01"]]))
assert debt_known_on("2024-01-31") is None
assert debt_known_on("2024-04-30")["value"] == 100
assert debt_known_on("2024-06-01")["value"] == 120

,id,entity,value,unit,period_end,available_at,ingested_at,source,supersedes
0,debt-v1,AAPL,100,USD million,2023-12-31,2024-02-02,2024-02-03,classroom/original,NaN
1,debt-v2,AAPL,120,USD million,2023-12-31,2024-05-10,2024-05-11,classroom/correction,debt-v1


,question_date,id,entity,value,unit,period_end,available_at,ingested_at,source,supersedes
0,2024-04-30,debt-v1,AAPL,100,USD million,2023-12-31,2024-02-02,2024-02-03,classroom/original,NaN
1,2024-06-01,debt-v2,AAPL,120,USD million,2023-12-31,2024-05-10,2024-05-11,classroom/correction,debt-v1


## 9. Choosing a memory store and retrieval method

A vector database is one possible component of memory. It does not decide which facts belong in memory, when they become available, or whether they support a conclusion.

| Approach | Useful for | Decisions still required |
|---|---|---|
| Files | Small working notes and procedures | Naming, concurrent edits, scope, versioning |
| SQL tables | Exact values, task state, dated evidence | Schema, transactions, indexes, migrations |
| Full-text search | Names, phrases, filing terminology | Tokenization, ranking, metadata filters |
| Vector search | Queries that use different words from the source | Embedding model, chunking, filters, evaluation |
| Hybrid retrieval | Combining lexical and semantic matches | Score combination, reranking, duplicate removal |

We use SQLite because students can inspect the state directly and the corpus is small. [SQLite FTS5](https://www.sqlite.org/fts5.html) provides a possible next step for full-text search. An embedding index becomes useful when literal wording is a poor guide to relevance, but similarity still does not establish the correct ticker, reporting period, or units.

Keep exact numbers and dates in structured fields even if their surrounding text enters a vector index. Retain the original source and a stable locator so a retrieved chunk can support a citation. If the embedding model or chunking policy changes, record that index version. Otherwise, a change in retrieval behavior becomes difficult to explain.

Chunking affects what retrieval can recover. Very small chunks may separate a value from its header or qualification. Very large chunks can consume the result budget with unrelated content. Overlap helps preserve context at boundaries, but can also return near-duplicate passages that crowd out a different source. Evaluate the retrieved evidence at the level the research question needs.

## 10. Retrieval is a pipeline

Our source retrieval first applies ticker and publication-date constraints, then ranks the eligible records, selects records that fit the response budget, and saves the returned evidence. The order matters.

The retrieval experiment below uses the same corpus as `06_memory_and_retrieval.py`. Its small corpus includes a future AAPL document and a document about MSFT. Both overlap strongly with the query “debt refinancing liquidity.” If we rank the entire corpus, take two records, and only then filter for AAPL as of April 30, we return nothing. The two slots have already gone to ineligible records. Filtering first leaves three eligible records, of which the debt and cash excerpts match the query.

The lexical baseline scores the number of distinct query terms found in the title and text. It uses deterministic tie-breaking so we can reproduce the result. This is intentionally simple enough to inspect line by line. Its weaknesses are visible: “rollover obligations” gets no matches even though the debt excerpt discusses refinancing.

Possible changes include a small domain vocabulary, stemming, query rewriting, or semantic retrieval. Each change should have a measurable purpose. If we tune the search on this one query and then report success on the same query, we have measured a repair to that example. We still need new queries to evaluate whether the improvement generalizes.

Response size changes the result too. The experiment fits one whole evidence record into 600 characters and both matches into 850. These budgets include the envelope and identifiers. The application search tool uses 12,000 characters and up to five records. Character counts are easy to enforce in this teaching example; actual token usage depends on the tokenizer and content.

A useful retrieval policy also considers coverage and disagreement. Five nearly identical excerpts may be less useful than a debt schedule, a cash disclosure, and a conflicting management statement. We should retain the provenance of conflicting claims and ask whether they refer to different periods or definitions. Picking the highest search score does not resolve the disagreement.

### Inspecting the corpus before ranking

The five excerpts make the failure visible. They include an AAPL document published after the question date, an MSFT document, and three eligible AAPL documents. The figures in this corpus are classroom examples. Inspect the titles and dates before calculating a search score.

In [14]:
# M4 retrieval ordering
query = "debt refinancing liquidity"
display(pd.DataFrame(corpus)[["record_id", "ticker", "available_on", "title"]])

ranked_globally = sorted(
    corpus, key=lambda record: -len(terms(query) & terms(record["title"] + " " + record["text"]))
)
wrong_order = [r for r in ranked_globally[:2]
               if r["ticker"] == "AAPL" and r["available_on"] <= "2024-04-30"]
eligible, ranked = rank_sources(corpus, ticker="AAPL", as_of="2024-04-30", query=query)

display(pd.DataFrame([
    {"policy": "Global top two", "records": [r["record_id"] for r in ranked_globally[:2]]},
    {"policy": "Then filter those two", "records": [r["record_id"] for r in wrong_order]},
    {"policy": "Filter first, then rank", "records": [r["record_id"] for r in ranked[:2]]},
]))
assert wrong_order == [] and {r["record_id"] for r in ranked[:2]} == {"aapl-debt", "aapl-cash"}

,record_id,ticker,available_on,title
0,aapl-future,AAPL,2024-05-10,Debt refinancing and liquidity update
1,msft-debt,MSFT,2024-02-02,Debt refinancing and liquidity
2,aapl-debt,AAPL,2024-02-02,Debt maturity schedule
3,aapl-cash,AAPL,2024-02-02,Cash resources
4,aapl-supply,AAPL,2024-04-10,Supplier concentration


,policy,records
0,Global top two,"[aapl-future, msft-debt]"
1,Then filter those two,[]
2,"Filter first, then rank","[aapl-debt, aapl-cash]"


In [15]:
# M5 whole-record budgets
candidates = [evidence_record("source", "AAPL", record) for record in ranked]
budget_results = []
for budget in [600, 850]:
    packed = pack_evidence(candidates, limit=2, max_chars=budget)
    serialized_size = len(json.dumps(packed, ensure_ascii=False))
    budget_results.append({
        "budget_chars": budget, "actual_chars": serialized_size,
        "records": [r["payload"]["record_id"] for r in packed["items"]],
        "matched": packed["matched"], "omitted": packed["omitted"],
    })
    assert serialized_size <= budget

display(pd.DataFrame(budget_results))
_, synonym_matches = rank_sources(corpus, ticker="AAPL", as_of="2024-04-30", query="rollover obligations")
print("Matches for 'rollover obligations':", len(synonym_matches))

,budget_chars,actual_chars,records,matched,omitted
0,600,433,[aapl-debt],2,1
1,850,806,"[aapl-debt, aapl-cash]",2,0


Matches for 'rollover obligations': 0


Try 400, 600, and 850 characters, then change the query to `liquidity`. Observe whether the first-ranked record fits and which evidence disappears. The omission count tells the caller that the response is incomplete. It does not explain whether an omitted record would have changed the conclusion, which is why evaluation must include the resulting research answer.

## 11. Context is a limited working area

Every model request has a context budget. Instructions, tool schemas, history, evidence, and the eventual answer compete for it. Here is an illustrative allocation for a 24,000-token design target:

| Component | Tokens |
|---|---:|
| Instructions | 2,000 |
| Tool definitions | 2,000 |
| Recent conversation | 6,000 |
| Retrieved evidence | 10,000 |
| Answer reserve | 2,000 |
| Headroom | 2,000 |

These are planning numbers, not measured usage from our demo. The provider's context capacity and output limits determine what is possible. We should inspect actual usage and leave room for the answer, rather than filling every available token with retrieved text.

There are several ways to manage a growing conversation. A sliding window retains recent exchanges but can drop the original reason for a decision. A summary can preserve that reason, but may omit an exception or alter a number. Retrieval brings back selected evidence, but can miss a relevant record. In practice, we often combine a short working summary with retrieval of exact records.

Our `SlidingWindowCompaction` trims older messages and preserves valid tool-call/result relationships. It does not generate an LLM summary. The configuration sets a 24,000-token trigger and a 12,000-token retention target, subject to the capability's other retention rules. Exact evidence remains in the application store and can be recalled. See [context compaction](https://pydantic.dev/docs/ai/harness/compaction/).

A useful working summary records the current question and scope, conclusions with evidence IDs, unresolved issues, and the next action. Keep exact financial values in retrievable evidence as well. If a summary says “leverage improved,” the underlying records should still tell us which leverage definition and dates support that statement.

Long histories also affect cost. Suppose every request repeats a fixed context of `C` tokens and each completed round adds `R` tokens. Across `n` requests, input tokens are approximately `n*C + R*n*(n-1)/2`, before compaction or provider caching. With `C=4000`, `R=1500`, and `n=8`, that is 74,000 input tokens. Reducing each addition to 500 tokens gives 46,000. These are token counts, not dollar prices. A provider cache may reduce billing for repeated content, but it does not improve the relevance of that content.

Retrieved documents and model-written memory are also inputs that can contain instructions. A source excerpt saying “ignore the research date” should remain quoted source content. It should not acquire the authority of the application's instructions. Keep project constraints in Python, identify source material clearly, and give tools only the access needed for their task.

In [16]:
# M6 context allocation
context_budget = pd.Series({
    "instructions": 2000, "tool definitions": 2000, "recent history": 6000,
    "retrieved evidence": 10000, "answer reserve": 2000, "headroom": 2000,
}, name="tokens")
display(context_budget.to_frame())
print("Allocated:", context_budget.sum())
print("After adding an unchanged 6,000-token excerpt:", context_budget.sum() + 6000)
assert context_budget.sum() == 24000

,tokens
instructions,2000
tool definitions,2000
recent history,6000
retrieved evidence,10000
answer reserve,2000
headroom,2000


Allocated: 24000
After adding an unchanged 6,000-token excerpt: 30000


**Pause:** which 6,000 tokens would you remove or defer? Give a reason tied to the current question. Removing an older tool result may be reasonable if its exact evidence remains retrievable. Removing the original scope without preserving it elsewhere can change the investigation.

In [17]:
# M7 repeated context cost
def repeated_input_tokens(n, fixed_context, added_per_round):
    return n * fixed_context + added_per_round * n * (n - 1) // 2

display(pd.DataFrame([
    {"requests": n, "1500 tokens added per round": repeated_input_tokens(n, 4000, 1500),
     "500 tokens added per round": repeated_input_tokens(n, 4000, 500)}
    for n in [1, 2, 4, 8]
]))
assert repeated_input_tokens(8, 4000, 1500) == 74000

,requests,1500 tokens added per round,500 tokens added per round
0,1,4000,4000
1,2,9500,8500
2,4,25000,19000
3,8,74000,46000


## 12. Concurrent edits, scope, and retention

M8 and M9 below use the actual `SqliteMemoryStore`. Two researchers read the same version of a note. Researcher A adds a cash comparison. Researcher B adds a refinancing question using the older version.

The store rejects B's stale write because its `expected_version` no longer matches. B then rereads the current note and deliberately merges the additional question. This is optimistic concurrency control: the write succeeds only if the record has not changed since we read it. The [memory capability](https://pydantic.dev/docs/ai/harness/memory/) exposes versioned memory operations.

Detecting the conflict prevents a lost update. It does not decide how to merge contradictory conclusions. If one researcher says debt increased and another says it decreased, concatenating their text preserves the conflict but does not resolve it. We must inspect periods, definitions, and evidence.

Scope is equally important. Our memory lives inside a project directory and uses a ticker namespace. A shared service would also need an authenticated user or organization scope supplied by the application. Allowing the model to choose an arbitrary namespace would weaken that boundary.

Retention should reflect the role of each record. We may discard an old scratch note after preserving its useful questions. A published report's citations need their supporting records to remain available, or a documented archive that can resolve them. Conversation transcripts, caches, and accepted evidence do not necessarily need the same retention period.

In [18]:
# M8 stale memory write
conflict_store = SqliteMemoryStore(database=fresh_path("memory-conflict.sqlite"))
memory_key = "AAPL/main/MEMORY.md"
await conflict_store.write(memory_key, "Check debt maturities.", expected_version=None)
writer_a = await conflict_store.read(memory_key, max_chars=8000)
writer_b = await conflict_store.read(memory_key, max_chars=8000)

await conflict_store.write(
    memory_key, writer_a.content + " Compare cash with near-term debt.", expected_version=writer_a.version
)
try:
    await conflict_store.write(
        memory_key, writer_b.content + " Inspect refinancing terms.", expected_version=writer_b.version
    )
except MemoryConflictError:
    print("Writer B's version is stale. Its write did not replace A's addition.")
else:
    raise AssertionError("Expected a version conflict.")

current = await conflict_store.read(memory_key, max_chars=8000)
print("Current memory:", current.content)

Writer B's version is stale. Its write did not replace A's addition.
Current memory: Check debt maturities. Compare cash with near-term debt.


In [19]:
# M9 merge against current version
current = await conflict_store.read(memory_key, max_chars=8000)
addition = " Inspect refinancing terms."
merged = current.content if addition in current.content else current.content + addition
await conflict_store.write(memory_key, merged, expected_version=current.version)
final_memory = await conflict_store.read(memory_key, max_chars=8000)
print(final_memory.content)
assert "near-term debt" in final_memory.content and "refinancing terms" in final_memory.content

Check debt maturities. Compare cash with near-term debt. Inspect refinancing terms.


We merged two compatible questions. If the notes made contradictory claims, preserving both sentences would only preserve the disagreement. We would still need to compare their definitions, dates, and supporting evidence. Version control tells us that the state changed; it does not decide which research conclusion is correct.

### Break: 10 minutes

After the break, we will compare two interruptions: losing the model connection after a settled calculation, and losing a write acknowledgement after the effect has committed.

## 13. What a checkpoint can restore

A checkpoint records a settled point in the agent conversation. On restart, our application looks for the latest usable snapshot from an earlier attempt and supplies its messages to a new run. The new attempt gets a new run ID while retaining the ticker's conversation identity.

The checkpoint does not contain every part of the application. Task status lives in the supervisor's database, working notes have their own store, and an external service has its own state. We need to understand the boundaries between these commits.

| Interruption point | What recovery can do |
|---|---|
| Before claiming a task | Leave it pending |
| After a calculation and settled tool result | Restore that result and continue |
| After evidence commit but before a settled tool result | Repeat the call and reuse the evidence ID |
| After a remote write with no acknowledgement | Reconcile the effect or reuse a remote idempotency key |
| After note commit but before task completion | Restart the attempt and validate a replacement note |
| After accepted completion | Skip the ticker on later invocations |

The recovery cell below runs `03_resume_research.py` in a fresh process. It interrupts the demo after a calculation, starts a fresh worker, and verifies that the settled calculation is reused. Then compare that result with the lost-acknowledgement experiment in script 07. They interrupt at different boundaries, so they require different mechanisms.

Saving steps gives us a basis for resumption. It does not guarantee that every operation executes exactly once. Durable execution frameworks can automate restart scheduling and replay, but writes still need appropriate idempotency or reconciliation. See [step persistence](https://pydantic.dev/docs/ai/harness/step-persistence/).

We also freeze the project configuration and hashes of its inputs, code, and dependency lock. Resuming old messages against changed calculation code can otherwise mix two meanings of the same tool. After changing inputs or code, create a new project. A larger application could introduce explicit state and tool-schema migrations, but that requires a compatibility policy.

### Recovery in a fresh Python worker

This experiment belongs in separate processes because that is the boundary we want to test. The script starts an agent, interrupts it after a settled calculation, and then starts the supervisor again. We inspect the durable result from this notebook.

**Before running:** how many attempts should the task have, and how many price calculations should appear in its ledger?

In [20]:
# R1 settled checkpoint recovery
recovery_project = fresh_path("recovery")
run_script("03_resume_research.py", recovery_project)
with contextlib.closing(connect(recovery_project)) as db:
    calculation_count = db.execute("SELECT count(*) FROM calls WHERE tool='analyze_prices'").fetchone()[0]
    starts = [json.loads(row[0]) for row in db.execute("SELECT payload FROM events WHERE kind='agent_started'")]
display(task_table(recovery_project))
display(pd.DataFrame(starts))
assert calculation_count == 1
assert task_table(recovery_project).iloc[0]["attempts"] == 2

Created Lecture 10/runs/notebook-6ddeb90e/recovery-14080b99. Run: python research.py run Lecture 10/runs/notebook-6ddeb90e/recovery-14080b99
Demonstration: model connection lost after the calculation

Starting the supervisor again. It will launch a fresh Python worker.
AAPL: attempt 2
COMPLETE: Lecture 10/runs/notebook-6ddeb90e/recovery-14080b99/report.html
Calculation calls across both attempts: 1
{'ticker': 'AAPL', 'status': 'complete', 'attempts': 2, 'error': None}



,ticker,status,attempts,error
0,AAPL,complete,2,None


,attempt,messages,resumed_from,ticker
0,1,0,NaN,AAPL
1,2,7,AAPL-attempt-1,AAPL


### The effect committed but the response disappeared

Now we move the interruption point. The naive writer commits a note and then raises a timeout. Repeating that function creates two notes. The timeout tells the caller that it lacks an acknowledgement, which leaves the outcome uncertain.

The following cells show the writes directly. They use an in-memory database so each run starts with an empty experiment.

In [21]:
# R2 duplicate effect
def make_effect_database():
    db = sqlite3.connect(":memory:")
    db.executescript("""
        CREATE TABLE naive_notes(id INTEGER PRIMARY KEY, body TEXT);
        CREATE TABLE checked_notes(id INTEGER PRIMARY KEY, body TEXT);
        CREATE TABLE operations(operation_key TEXT PRIMARY KEY, body TEXT, receipt TEXT);
    """)
    return db


def naive_write(db, note):
    with db:
        db.execute("INSERT INTO naive_notes(body) VALUES(?)", (note,))
    raise TimeoutError("The acknowledgement was lost after commit.")


effect_db = make_effect_database()
note_text = "Investigate AAPL's debt maturities."
for attempt in range(2):
    try:
        naive_write(effect_db, note_text)
    except TimeoutError as error:
        print("Attempt", attempt + 1, str(error))

display(pd.read_sql_query("SELECT * FROM naive_notes", effect_db))
assert effect_db.execute("SELECT count(*) FROM naive_notes").fetchone()[0] == 2

Attempt 1 The acknowledgement was lost after commit.
Attempt 2 The acknowledgement was lost after commit.


,id,body
0,1,Investigate AAPL's debt maturities.
1,2,Investigate AAPL's debt maturities.


In [22]:
# R3 effect and receipt
def save_once(db, operation_key, note, *, lose_reply=False):
    with db:
        db.execute("BEGIN IMMEDIATE")
        previous = db.execute(
            "SELECT body,receipt FROM operations WHERE operation_key=?", (operation_key,)
        ).fetchone()
        if previous:
            if previous[0] != note:
                raise ValueError("Operation key was reused with different arguments.")
            receipt = json.loads(previous[1])
        else:
            row = db.execute("INSERT INTO checked_notes(body) VALUES(?)", (note,))
            receipt = {"note_id": row.lastrowid}
            db.execute("INSERT INTO operations VALUES(?,?,?)", (operation_key, note, json.dumps(receipt)))
    if lose_reply:
        raise TimeoutError("The acknowledgement was lost after commit.")
    return receipt

operation_key = "project-17:AAPL:debt-question:v1"
try:
    save_once(effect_db, operation_key, note_text, lose_reply=True)
except TimeoutError:
    print("No acknowledgement. Retrying the same operation.")
receipt = save_once(effect_db, operation_key, note_text)
display(receipt)
display(pd.read_sql_query("SELECT * FROM checked_notes", effect_db))
assert effect_db.execute("SELECT count(*) FROM checked_notes").fetchone()[0] == 1

try:
    save_once(effect_db, operation_key, "A different research note.")
except ValueError as error:
    print(type(error).__name__ + ":", error)

No acknowledgement. Retrying the same operation.


{'note_id': 1}

,id,body
0,1,Investigate AAPL's debt maturities.


ValueError: Operation key was reused with different arguments.


The receipt and the note share one SQLite transaction. For a remote write, a local receipt cannot make the remote service's commit atomic with ours. We would need the service's idempotency support or a way to inspect and reconcile the prior effect. The checkpoint experiment and this experiment solve different problems because they preserve different state.

## 14. Budgets, failure categories, and progress

The defaults are two attempts per ticker, twelve model requests per attempt, twenty successful agent tool calls per attempt, and a 120-second worker deadline. Planning and memory calls count toward the successful-call limit. The application separately reserves dispatched research actions, including ones that fail after dispatch. `run --seconds` limits a supervisor invocation, and attempt counts persist across invocations.

These limits control different failure modes. Request limits bound a model that repeatedly proposes invalid arguments. Tool limits constrain successful activity. A worker deadline bounds a process that does not return. A retry limit bounds repeated attempts across restarts. A fresh process should not reset the whole project's budget.

| Failure | Useful response |
|---|---|
| Repairable argument or citation | Return specific validation feedback |
| Missing required price data | Record the failure and preserve the other tickers |
| Temporary provider failure | Retry within a bounded policy |
| Rate limit | Respect the service's retry timing and total deadline |
| Uncertain external write | Check the prior effect before repeating it |
| Repeated calls with no new evidence | Stop, revise the question, or report the unresolved issue |

Retries can multiply. Two task attempts, three SDK attempts per request, and three HTTP attempts inside a tool can permit eighteen underlying HTTP attempts. Decide which layer owns each retry and observe the total cost. Add backoff and jitter for transient service failures, but do not use repeated retries to conceal a deterministic data error.

Our worker uses a subprocess deadline so the supervisor can terminate work that outlasts its budget. Cancellation cannot undo an external operation that already committed. A forcibly terminated process may also miss its final usage receipt, although earlier committed events remain available.

Progress deserves a definition beyond “the model is still making calls.” New evidence, resolution of an open question, and acceptance of a valid result are useful signals. Rewriting the same plan repeatedly consumes resources without necessarily advancing the investigation. A longer research application could track repeated tool arguments and unchanged evidence sets, then request a revised plan or stop with an explanation.

In [23]:
# R4 request budget
from pydantic_ai.exceptions import UsageLimitExceeded

successful_pings = []
def repeat_forever(messages, info):
    return ModelResponse(parts=[ToolCallPart("ping", {})])

budget_agent = Agent(FunctionModel(repeat_forever))

@budget_agent.tool_plain
def ping() -> str:
    """Record a successful call without adding research evidence."""
    successful_pings.append("ping")
    return "No new evidence."

try:
    await budget_agent.run("Continue researching.", usage_limits=UsageLimits(request_limit=2, tool_calls_limit=10))
except UsageLimitExceeded as error:
    print(type(error).__name__ + ":", error)
print("Successful pings before stopping:", len(successful_pings))
assert len(successful_pings) == 2

UsageLimitExceeded: The next request would exceed the request_limit of 2. Consider raising the limit, or see the docs on usage limits for budget-aware patterns: https://pydantic.dev/docs/ai/core-concepts/agent/#usage-limits
Successful pings before stopping: 2


## 15. Plans and research decomposition

A plan is useful when the model must keep track of several related questions. For a refinancing investigation, the plan might ask for the debt schedule, compare near-term obligations with cash, inspect qualifications in the disclosure, and draft a conclusion with citations.

A plan should include a recognizable completion condition. “Research AAPL” is difficult to evaluate. “Produce a dated risk calculation and a source-supported interpretation, with both cited in the final note” is closer to a contract we can check.

The application queue and the model's plan operate at different levels. The queue assigns tickers and attempts. The plan organizes work within one ticker. Splitting work across more agents would add coordination questions: who owns shared evidence, how conflicting conclusions are resolved, which tasks depend on others, and who accepts the final report. Add that complexity when the task structure justifies it.

We can test whether planning helps by comparing repeated provider-backed runs with and without the capability. Compare supported conclusions, missed evidence, latency, and cost. The deterministic demo follows a fixed sequence, so it cannot establish that planning improves research quality.

In [24]:
# R5 resume the portfolio project
run_script("research.py", "run", project)
before_repeat = task_table(project)
run_script("research.py", "run", project)
after_repeat = task_table(project)
display(after_repeat)
assert after_repeat.status.eq("complete").all()
pd.testing.assert_frame_equal(before_repeat, after_repeat)
assert after_repeat.attempts.eq(1).all()

JPM: attempt 1
MSFT: attempt 1
COMPLETE: Lecture 10/runs/notebook-6ddeb90e/portfolio-eeebdf7a/report.html



COMPLETE: Lecture 10/runs/notebook-6ddeb90e/portfolio-eeebdf7a/report.html



,ticker,status,attempts,error
0,AAPL,complete,1,None
1,JPM,complete,1,None
2,MSFT,complete,1,None


## 16. Acceptance and report generation

The model returns a typed `ResearchNote` containing a body and citation IDs. Pydantic checks the structure. Our validator checks that cited records exist, belong to the assigned ticker, and include both calculated metrics and source evidence. Invalid references return to the model for repair. Python saves an accepted note only after these checks. See [structured outputs](https://pydantic.dev/docs/ai/output/).

Citation validity is only one layer of review. A real price-volatility record does not support the sentence “earnings will grow next quarter.” The reference can exist and still fail to support the claim. A future extension could represent individual claims with their supporting IDs, making claim-level review easier than reviewing one large paragraph.

The report builds numerical tables from saved calculations and adds accepted interpretations. It includes actual data coverage, task status, and an evidence appendix. A failed ticker remains visible. Portfolio weights require prices for every holding because the denominator includes all equity market values plus cash. Silently dropping a failed holding would change the meaning of every reported weight.

A report that compares tickers should also make their observation windows visible. If one stock has a shorter history, apparent risk differences may partly reflect different periods. The current report exposes coverage so the reader can identify that issue. A stricter comparison module could require a common window before calculating comparative statistics.

In [25]:
# P1 inspect the report
summary = json.loads((project / "summary.json").read_text())
report_evidence = json.loads((project / "evidence.json").read_text())
display(pd.DataFrame(summary["metrics"]).T)
display(Markdown((project / "report.md").read_text()))
assert summary["status"] == "COMPLETE"
print("HTML report:", (project / "report.html").relative_to(LECTURE_DIR))

,annualized_log_volatility,approximate_sharpe_rf_2pct,conventions,first_date,last_date,last_raw_close,max_drawdown,mean_rolling_20_log_volatility,observations,requested_as_of,requested_start,source_sha256,total_return
AAPL,0.302859,0.069976,Daily log returns; sample std; 252 sessions; 2% annual risk-free approximation,2024-01-02,2024-04-30,101.399142,-0.115432,0.297494,86,2024-04-30,2024-01-02,f299641567e3b744855d18e5f91217b17db0f0294e08d3dc7074d6dd56ac8481,0.013991
JPM,0.141896,-3.376732,Daily log returns; sample std; 252 sessions; 2% annual risk-free approximation,2024-01-02,2024-04-30,51.391579,-0.159744,0.141542,86,2024-04-30,2024-01-02,f299641567e3b744855d18e5f91217b17db0f0294e08d3dc7074d6dd56ac8481,-0.143474
MSFT,0.182094,1.68241,Daily log returns; sample std; 252 sessions; 2% annual risk-free approximation,2024-01-02,2024-04-30,89.309461,-0.115935,0.183252,86,2024-04-30,2024-01-02,f299641567e3b744855d18e5f91217b17db0f0294e08d3dc7074d6dd56ac8481,0.116368


# Stock research report

**COMPLETE** · Generated sample data

Question: How concentrated is the portfolio, and how do its holdings compare on historical risk?

Requested window: 2024-01-02 through 2024-04-30.
Engine: demo. Completed 3 of 3 tickers. Research actions: 12.

Recorded model requests: 21. Successful agent tool calls: 18. Usage receipts: 3 of 3 attempts.

## Computed evidence

| Ticker | First date | Last date | Prices | Total return | Ann. log vol. | Max drawdown |
| --- | --- | --- | --- | --- | --- | --- |
| AAPL | 2024-01-02 | 2024-04-30 | 86 | 1.40% | 30.29% | -11.54% |
| JPM | 2024-01-02 | 2024-04-30 | 86 | -14.35% | 14.19% | -15.97% |
| MSFT | 2024-01-02 | 2024-04-30 | 86 | 11.64% | 18.21% | -11.59% |

Returns and drawdown use adjusted prices. Annualized volatility uses the sample standard deviation of daily log returns and 252 sessions per year. The dates and row counts show the history available for each stock.

AAPL has the highest annualized volatility over the shared date range.

## Portfolio exposure

Valuation date: 2024-04-30. Total value including cash: 19,526.01. Cash: 5,000.00 (25.61%). Values use the supplied price units.

| Ticker | Shares | Weighted entry | Raw close | Market value | Weight | Unrealized P&L |
| --- | --- | --- | --- | --- | --- | --- |
| AAPL | 60 | 95.00 | 101.40 | 6,083.95 | 31.16% | 383.95 |
| JPM | 60 | 55.00 | 51.39 | 3,083.49 | 15.79% | -216.51 |
| MSFT | 60 | 76.17 | 89.31 | 5,358.57 | 27.44% | 788.57 |

Each weight is the holding's market value divided by total portfolio value, including cash.

## Research notes


### AAPL

AAPL returned 1.40% from 2024-01-02 to 2024-04-30. Its largest decline from a previous peak was 11.54%. Next for AAPL: compare the latest eligible filing's business risks with the price history.

Evidence: 94fd689e4b31e5167fb5, e8bcca78a5f069b81059

### JPM

JPM returned -14.35% from 2024-01-02 to 2024-04-30. Its largest decline from a previous peak was 15.97%. Next for JPM: compare the latest eligible filing's business risks with the price history.

Evidence: 5e4457a233784a9c211d, 9eaacb15788f5c75ca24

### MSFT

MSFT returned 11.64% from 2024-01-02 to 2024-04-30. Its largest decline from a previous peak was 11.59%. Next for MSFT: compare the latest eligible filing's business risks with the price history.

Evidence: 424584a37937812b2e46, 45769c580b06b56161bb

## Task status


AAPL: complete, attempts=1, error=none

JPM: complete, attempts=1, error=none

MSFT: complete, attempts=1, error=none

## Project provenance

```json
{
  "contract_version": 2,
  "question": "How concentrated is the portfolio, and how do its holdings compare on historical risk?",
  "dataset": "fixture",
  "start": "2024-01-02",
  "as_of": "2024-04-30",
  "engine": "demo",
  "model": null,
  "code_sha256": "9d253c3c69887d38a23c21f7dbc0cbcfe005c3ceaf754a63dcd6546482db016d",
  "data_sha256": "f299641567e3b744855d18e5f91217b17db0f0294e08d3dc7074d6dd56ac8481",
  "sources_sha256": "8e6a9503c22b119ab3889f34824ecb918bdec8ca3038c0ea7cc11c9876da9a84",
  "holdings_sha256": "3da79d023ad6e0059e6043675b18f9abf03cf8202bf5bab8758a370d28198ce1"
}
```

## Evidence appendix

See evidence.json for the complete payloads and source hashes.

### 94fd689e4b31e5167fb5 · metrics · AAPL

```json
{
  "annualized_log_volatility": 0.30285860900450096,
  "approximate_sharpe_rf_2pct": 0.0699763454866972,
  "conventions": "Daily log returns; sample std; 252 sessions; 2% annual risk-free approximation",
  "first_date": "2024-01-02",
  "last_date": "2024-04-30",
  "last_raw_close": 101.399142,
  "max_drawdown": -0.11543233728107127,
  "mean_rolling_20_log_volatility": 0.2974935160025936,
  "observations": 86,
  "requested_as_of": "2024-04-30",
  "requested_start": "2024-01-02",
  "source_sha256": "f299641567e3b744855d18e5f91217b17db0f0294e08d3dc7074d6dd56ac8481",
  "total_return": 0.013991419999999977
}
```

### e8bcca78a5f069b81059 · source · AAPL

```json
{
  "available_on": "2024-01-02",
  "locator": "data/prices.csv",
  "source_sha256": "8e6a9503c22b119ab3889f34824ecb918bdec8ca3038c0ea7cc11c9876da9a84",
  "source_type": "synthetic teaching note",
  "text": "All prices are generated teaching values. The calendar contains weekdays, including US market holidays. Raw and adjusted prices are equal by construction. These are not issuer financial results or historical market prices.",
  "ticker": "AAPL",
  "title": "AAPL synthetic data construction note"
}
```

### 5e4457a233784a9c211d · metrics · JPM

```json
{
  "annualized_log_volatility": 0.1418960015264686,
  "approximate_sharpe_rf_2pct": -3.3767315084591334,
  "conventions": "Daily log returns; sample std; 252 sessions; 2% annual risk-free approximation",
  "first_date": "2024-01-02",
  "last_date": "2024-04-30",
  "last_raw_close": 51.391579,
  "max_drawdown": -0.1597442280502146,
  "mean_rolling_20_log_volatility": 0.14154240472972857,
  "observations": 86,
  "requested_as_of": "2024-04-30",
  "requested_start": "2024-01-02",
  "source_sha256": "f299641567e3b744855d18e5f91217b17db0f0294e08d3dc7074d6dd56ac8481",
  "total_return": -0.14347368333333332
}
```

### 9eaacb15788f5c75ca24 · source · JPM

```json
{
  "available_on": "2024-01-02",
  "locator": "data/prices.csv",
  "source_sha256": "8e6a9503c22b119ab3889f34824ecb918bdec8ca3038c0ea7cc11c9876da9a84",
  "source_type": "synthetic teaching note",
  "text": "All prices are generated teaching values. The calendar contains weekdays, including US market holidays. Raw and adjusted prices are equal by construction. These are not issuer financial results or historical market prices.",
  "ticker": "JPM",
  "title": "JPM synthetic data construction note"
}
```

### 424584a37937812b2e46 · metrics · MSFT

```json
{
  "annualized_log_volatility": 0.1820942114698025,
  "approximate_sharpe_rf_2pct": 1.682410300548962,
  "conventions": "Daily log returns; sample std; 252 sessions; 2% annual risk-free approximation",
  "first_date": "2024-01-02",
  "last_date": "2024-04-30",
  "last_raw_close": 89.309461,
  "max_drawdown": -0.11593514846845998,
  "mean_rolling_20_log_volatility": 0.1832523229865978,
  "observations": 86,
  "requested_as_of": "2024-04-30",
  "requested_start": "2024-01-02",
  "source_sha256": "f299641567e3b744855d18e5f91217b17db0f0294e08d3dc7074d6dd56ac8481",
  "total_return": 0.11636826249999999
}
```

### 45769c580b06b56161bb · source · MSFT

```json
{
  "available_on": "2024-01-02",
  "locator": "data/prices.csv",
  "source_sha256": "8e6a9503c22b119ab3889f34824ecb918bdec8ca3038c0ea7cc11c9876da9a84",
  "source_type": "synthetic teaching note",
  "text": "All prices are generated teaching values. The calendar contains weekdays, including US market holidays. Raw and adjusted prices are equal by construction. These are not issuer financial results or historical market prices.",
  "ticker": "MSFT",
  "title": "MSFT synthetic data construction note"
}
```


HTML report: runs/notebook-6ddeb90e/portfolio-eeebdf7a/report.html


### A citation that exists can still fail to support a claim

We can check a note's structure and citations in Python. The following example deliberately makes an earnings claim while citing historical price-risk evidence and its data-construction source. It passes the existing reference checks. Read the claim against the evidence and decide what would be needed to support it.

The example calls the validator without saving this note to the project.

In [26]:
# P2 citation versus support
from research_harness.tools import validate_note

with contextlib.closing(connect(project)) as db:
    saved = db.execute("SELECT citations FROM notes WHERE ticker='JPM'").fetchone()
    citations = json.loads(saved[0])
    unsupported_claim = "JPM's historical volatility shows that its earnings will rise next quarter."
    validate_note(db, "JPM", unsupported_claim, citations)
    cited_records = [r for r in evidence(db, "JPM") if r["id"] in citations]
print("The citation checks pass.")
print("Claim:", unsupported_claim)
display(pd.DataFrame(cited_records)[["id", "kind", "payload"]])

The citation checks pass.
Claim: JPM's historical volatility shows that its earnings will rise next quarter.


,id,kind,payload
0,5e4457a233784a9c211d,metrics,"{'annualized_log_volatility': 0.1418960015264686, 'approximate_sharpe_rf_2pct': -3.3767315084591..."
1,9eaacb15788f5c75ca24,source,"{'available_on': '2024-01-02', 'locator': 'data/prices.csv', 'source_sha256': '8e6a9503c22b119ab..."


## 17. Evaluating the harness in layers

An end-to-end report can look plausible even when retrieval missed the key source. Evaluate the components at the level where failures arise.

| Layer | Example check |
|---|---|
| Calculation | Prices 100, 110, 99 produce −1% total return |
| Tool interface | A schema failure never enters the calculation |
| Retrieval | Correct ticker and date, with the required supporting excerpt |
| Memory | A stale edit cannot silently overwrite a newer finding |
| Recovery | An uncertain retry does not duplicate the intended write |
| Output | Citation IDs resolve to the assigned ticker's evidence |
| Claim support | Each substantive claim follows from its cited evidence |
| Whole project | Complete coverage, useful report, acceptable latency and cost |

The evaluation cells below compare the same policies as `08_evaluate_retrieval.py` and its `--rewrite` option. The evaluation has four questions with known supporting excerpts. Recall@2 is the fraction of expected relevant records found among the top two returned results. Precision here is the fraction of returned records that are relevant, using zero when nothing returns.

The lexical baseline has mean recall of 0.75. Rewriting “rollover obligations” as “debt refinancing” raises it to 1.0 on these four examples. We have repaired one known miss. To evaluate the policy, add held-out questions, ambiguous terms, duplicate excerpts, conflicting statements, and documents from the wrong date or ticker. A metadata violation should fail the evaluation even if the returned passage sounds highly relevant.

Compare configurations under the same corpus, questions, budgets, and acceptance rules. For a provider-backed model, use repeated runs and retain trajectories so we can inspect variation. Track token usage and elapsed time alongside quality. A retrieval change that improves recall but floods the model with irrelevant text may worsen the final answer or cost more than the improvement warrants.

In [27]:
# P3 retrieval evaluation
cases = [
    ("debt refinancing", {"aapl-debt"}),
    ("liquidity", {"aapl-cash"}),
    ("supplier concentration", {"aapl-supply"}),
    ("rollover obligations", {"aapl-debt"}),
]

def evaluate_retrieval(rewrite=False):
    rows = []
    for question, expected in cases:
        query = "debt refinancing" if rewrite and question == "rollover obligations" else question
        _, ranked = rank_sources(corpus, ticker="AAPL", as_of="2024-04-30", query=query)
        returned = {r["record_id"] for r in ranked[:2]}
        relevant = returned & expected
        rows.append({
            "question": question, "query": query, "returned": sorted(returned),
            "recall_at_2": len(relevant) / len(expected),
            "precision_among_returned": len(relevant) / len(returned) if returned else 0,
        })
    return pd.DataFrame(rows)

baseline = evaluate_retrieval()
rewritten = evaluate_retrieval(rewrite=True)
display(baseline)
display(pd.Series({"baseline": baseline.recall_at_2.mean(),
                   "one known rewrite": rewritten.recall_at_2.mean()}, name="mean recall@2").to_frame())
assert baseline.recall_at_2.mean() == .75 and rewritten.recall_at_2.mean() == 1.0

,question,query,returned,recall_at_2,precision_among_returned
0,debt refinancing,debt refinancing,[aapl-debt],1.0,1.0
1,liquidity,liquidity,[aapl-cash],1.0,1.0
2,supplier concentration,supplier concentration,[aapl-supply],1.0,1.0
3,rollover obligations,rollover obligations,[],0.0,0.0


,mean recall@2
baseline,0.75
one known rewrite,1.00


Add two new questions before changing the policy again. Keep their expected supporting IDs separate from the examples used to tune the search. Also inspect any result with the wrong ticker or publication date as a hard failure. A relevant-looking passage can still be ineligible for the question.

## 18. Adding a research module

Suppose we want a leverage tool. First define the question precisely: debt divided by which equity measure, for which period, using what units? Decide what to return when equity is zero or negative. Determine which publication date makes the inputs eligible.

Then work through the same boundaries we have studied:

1. Define model arguments and application-owned scope.
2. Validate source fields and calculate the ratio in Python.
3. Save the inputs, units, dates, calculation version, and provenance with the result.
4. Specify retry behavior and any cache key.
5. Add acceptance checks and a report section that can explain the result.
6. Evaluate numerical cases, unavailable data, and claims that overstate what the ratio shows.

A notebook can remain behind a tool when it already contains useful analysis. A Papermill adapter would need a parameter schema, an isolated output directory, a deadline, and a structured result extracted from the executed notebook. The model-facing contract should describe the research operation rather than notebook execution details.

### A benchmark module with a checked calendar

Here is a smaller extension we can implement completely. It compares endpoint returns over a shared set of dates. The calculation receives observations supplied by Python, checks their calendars and prices, and returns both returns plus their difference in percentage points.

This comparison is a price-return difference over a fixed window. It does not estimate alpha from a regression or adjust for risk. Those would require different definitions and inputs.

In [28]:
# X1 benchmark calculation
class BenchmarkComparison(BaseModel):
    first_date: str
    last_date: str
    stock_return: float
    benchmark_return: float
    difference_percentage_points: float


def compare_returns(stock: pd.Series, benchmark: pd.Series) -> BenchmarkComparison:
    if not stock.index.equals(benchmark.index):
        raise ValueError("Stock and benchmark must use the same observation dates.")
    if len(stock) < 2 or not stock.index.is_unique or not stock.index.is_monotonic_increasing:
        raise ValueError("Use at least two unique dates in increasing order.")
    values = np.column_stack([stock.to_numpy(dtype=float), benchmark.to_numpy(dtype=float)])
    if not np.isfinite(values).all() or (values <= 0).any():
        raise ValueError("Prices must be finite and positive.")
    stock_return = float(stock.iloc[-1] / stock.iloc[0] - 1)
    benchmark_return = float(benchmark.iloc[-1] / benchmark.iloc[0] - 1)
    return BenchmarkComparison(
        first_date=str(stock.index[0].date()), last_date=str(stock.index[-1].date()),
        stock_return=stock_return, benchmark_return=benchmark_return,
        difference_percentage_points=100 * (stock_return - benchmark_return),
    )

dates = pd.to_datetime(["2024-04-29", "2024-04-30"])
stock_prices = pd.Series([100.0, 102.0], index=dates)
benchmark_prices = pd.Series([200.0, 202.0], index=dates)
comparison = compare_returns(stock_prices, benchmark_prices)
display(comparison.model_dump())
assert np.isclose(comparison.difference_percentage_points, 1.0)

try:
    compare_returns(stock_prices, benchmark_prices.set_axis(dates + pd.Timedelta(days=1)))
except ValueError as error:
    print("Calendar check:", error)

{'first_date': '2024-04-29',
 'last_date': '2024-04-30',
 'stock_return': 0.020000000000000018,
 'benchmark_return': 0.010000000000000009,
 'difference_percentage_points': 1.0000000000000009}

Calendar check: Stock and benchmark must use the same observation dates.


### Registering the module as a tool

The model requests the comparison, while Python supplies both series through dependencies. The tool returns the typed comparison. This gives us a complete agent example for the new module. To add it to the persistent stock-research application, we would also save its input hashes and result as evidence, then extend the final-note checks and report.

In [29]:
# X2 benchmark agent
@dataclass
class BenchmarkDeps:
    stock: pd.Series
    benchmark: pd.Series


def benchmark_model(messages, info):
    observations = [p for m in messages for p in m.parts if isinstance(p, ToolReturnPart)]
    if not observations:
        return ModelResponse(parts=[ToolCallPart("compare_with_benchmark", {})])
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, observations[-1].content)])

benchmark_agent = Agent(
    FunctionModel(benchmark_model), deps_type=BenchmarkDeps, output_type=BenchmarkComparison
)

@benchmark_agent.tool
def compare_with_benchmark(ctx: RunContext[BenchmarkDeps]) -> dict:
    """Compare endpoint returns over the application's aligned date window."""
    return compare_returns(ctx.deps.stock, ctx.deps.benchmark).model_dump()

benchmark_result = await benchmark_agent.run(
    "Compare the stock with its benchmark.", deps=BenchmarkDeps(stock_prices, benchmark_prices)
)
display(benchmark_result.output.model_dump())

{'first_date': '2024-04-29',
 'last_date': '2024-04-30',
 'stock_return': 0.020000000000000018,
 'benchmark_return': 0.010000000000000009,
 'difference_percentage_points': 1.0000000000000009}

## 19. Optional runs with course data or a model provider

The course-data branch uses the Lecture 4 database and the same three holdings. Set `RUN_COURSE` to `True` after placing `data.db` in the Lecture 4 data folder. The provider branch uses the same persistent harness as the demo. Set `RUN_PROVIDER` to `True`, supply an available model ID, and configure `ANTHROPIC_API_KEY` in the environment that launches Jupyter.

These switches stay off during a normal Run All. Provider execution uses the configured API account. Set it deliberately when you want the model to choose its own tool sequence. The application retains the same validation and budgets. See the [provider setup](readme.md#use-a-model-provider).

In [30]:
# X3 optional configuration
RUN_COURSE = False
RUN_PROVIDER = False
PROVIDER_MODEL = ""  # For example, an available anthropic:... model ID.

In [31]:
# X4 course database
if RUN_COURSE:
    course_project = fresh_path("course")
    run_script("research.py", "init", course_project, "--dataset", "course", "--as-of", "2024-09-30")
    run_script("research.py", "run", course_project, timeout=450)
    course_summary = json.loads((course_project / "summary.json").read_text())
    display(pd.DataFrame(course_summary["metrics"]).T)
else:
    print("Course database run is off. Set RUN_COURSE=True to execute it.")

Course database run is off. Set RUN_COURSE=True to execute it.


In [32]:
# X5 provider project
if RUN_PROVIDER:
    if not PROVIDER_MODEL.startswith("anthropic:") or not os.environ.get("ANTHROPIC_API_KEY"):
        raise ValueError("Choose an Anthropic model ID and set ANTHROPIC_API_KEY before running this cell.")
    provider_project = fresh_path("provider")
    run_script("research.py", "init", provider_project, "--engine", "model", "--model", PROVIDER_MODEL)
    run_script("research.py", "run", provider_project, "--max-tasks", "1", expected=(0, 2), timeout=150)
    display(task_table(provider_project))
    print("Continue with: python research.py run", provider_project.relative_to(LECTURE_DIR))
else:
    print("Provider run is off. Set the configuration explicitly to execute it.")

Provider run is off. Set the configuration explicitly to execute it.


## 20. Practice and discussion

Choose one exercise and write enough code or a record schema that another person can test your design.

**A. A correction in memory.** Add another reporting period to the debt example. A query should choose the correct version within each period without mixing periods. Show what an April and a June investigation can retrieve.

**B. Retrieval under pressure.** Add a synonym query and a near-duplicate excerpt. Change one retrieval policy and evaluate both the original questions and two held-out questions. Explain whether the change improved coverage or only the example you tuned it on.

**C. An uncertain write.** Use a different operation key on every retry in R3. Predict the number of records. Then reuse one key with different arguments and explain why the writer rejects it.

**D. Context allocation.** Make room for another 6,000-token excerpt within M6's 24,000-token design target. State which information you will summarize, remove, or retrieve later, and what question that choice could make harder to answer.

**E. A benchmark module.** Extend X1 to include the source hashes and calculation version. Add a case with different calendars. Explain the difference between the return difference, a portfolio weight, and a risk-adjusted performance estimate.

**F. Claim support.** Rewrite P2's earnings conclusion so it follows from the available evidence. Then list the additional business evidence needed for the original claim. Which parts of this review could you automate?

**G. Planning.** Compare repeated provider-backed runs with and without planning, keeping questions, corpus, and budgets fixed. Track supported claims, evidence coverage, cost, and latency. Explain why the fixed demo cannot answer the quality question.

The project report, evidence payloads, logs, and capability stores remain under the path printed in S0. The Python package can run the same investigation from a terminal after the notebook kernel closes.